In [ ]:
import numpy as np
import pandas as pd
import joblib
from hmmlearn.hmm import GaussianHMM
from sklearn.model_selection import train_test_split

ACTION_MAP = {
    'воспользовался поиском': 'SEARCH',
    'воспользовался каталогом': 'CATALOG',
    'находится в карточке товара': 'CARD_VIEW',
    'просматривает сертификат товара': 'CARD_VIEW',
    'добавил в корзину': 'CART_VIEW',
    'находится в корзине': 'CART_VIEW',
    'оформление заказа': 'CART_VIEW',
    'подтверждение заказа': 'CART_VIEW',
    'оплата': 'CART_VIEW',
    'находится на странице акций': 'PROMO',
    'нажал применить промокод': 'PROMO',
    'рассрочка': 'PROMO',
    'распродажа': 'PROMO',
    'цены снижены': 'PROMO',
    'находится в списке статей': 'ARTICLES',
    'находится на главной странице': 'MAIN',
    'личный кабинет': 'CABINET',
    'избранное': 'CABINET',
}

def map_action(text):
    t = str(text).lower()
    for key, cls in ACTION_MAP.items():
        if key in t:
            return cls
    return 'OTHER'

def build_features(df):
    df = df.copy()
    df.sort_values(['sess_id', 'watch_tms'], inplace=True)
    df['action'] = df['goal_nm_lvl1'].apply(map_action)
    df['watch_tms'] = pd.to_datetime(df['watch_tms'])
    df['delta_sec'] = df.groupby('sess_id')['watch_tms'].diff().dt.total_seconds().fillna(0)
    df['delta_sec'] = df['delta_sec'].clip(0, 3600)
    df['step_idx'] = df.groupby('sess_id').cumcount()
    df['sess_len'] = df.groupby('sess_id')['action'].transform('size')
    df['unique_actions'] = df.groupby('sess_id')['action'].transform('nunique')
    df['sess_time'] = df.groupby('sess_id')['delta_sec'].transform('sum')
    df['time_from_start'] = df.groupby('sess_id')['delta_sec'].cumsum()
    df['time_to_end'] = df['sess_time'] - df['time_from_start']
    df['is_cart'] = df['goal_nm_lvl1'].str.contains('корзин', case=False, na=False)
    df['act_id'] = pd.factorize(df['action'])[0]
    feat_cols = [
        'act_id', 'delta_sec',
        'step_idx', 'sess_len', 'unique_actions',
        'sess_time', 'time_from_start', 'time_to_end'
    ]
    return df, feat_cols

df = pd.read_csv("../data/actions.csv", sep=";")
df_feat, feat_cols = build_features(df)

session_target = df_feat.groupby('sess_id')['is_cart'].max().astype(int)
sessions = df_feat['sess_id'].unique()

train_sess, test_sess = train_test_split(sessions, test_size=0.2, random_state=42)

def get_sequences(df, session_ids):
    sequences = []
    for s in session_ids:
        seq = df[df['sess_id'] == s][feat_cols].values
        if len(seq) > 1:
            sequences.append(seq)
    return sequences

train_sequences = get_sequences(df_feat, train_sess)

pos_sequences = [seq for s, seq in zip(train_sess, train_sequences) if session_target[s] == 1]
neg_sequences = [seq for s, seq in zip(train_sess, train_sequences) if session_target[s] == 0]

def train_hmm(sequences):
    lengths = [len(s) for s in sequences]
    X = np.vstack(sequences)
    model = GaussianHMM(n_components=4, covariance_type='diag', n_iter=100)
    model.fit(X, lengths)
    return model

hmm_pos = train_hmm(pos_sequences)
hmm_neg = train_hmm(neg_sequences)

def session_score(seq):
    ll_pos = hmm_pos.score(seq)
    ll_neg = hmm_neg.score(seq)
    return 1 / (1 + np.exp(-(ll_pos - ll_neg)))

print(session_score(train_sequences[0]))

joblib.dump(hmm_pos, "../models/hmm_pos.pkl")
joblib.dump(hmm_neg, "../models/hmm_neg.pkl")